<a href="https://colab.research.google.com/github/Luizadraeger/URBAN-DATA-ANALYTICS/blob/main/pipeline_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Célula 3: Importa livrarias
import ee
import geemap
import geopandas as gpd
import os
import time
import pandas as pd

In [3]:
# Célula 4: Autenticação e inicialização dos serviços Google
# Esta célula configura o acesso ao Google Drive e ao Google Earth Engine (EE).

# PASSOS NECESSÁRIOS:
# 1. Monte o Google Drive para leitura/escrita de arquivos.
# 2. Autentique sua conta Google para uso do Earth Engine.
# 3. Inicialize o Earth Engine com um projeto válido do Google Cloud.

# IMPORTANTE:
# - Antes de executar, crie um projeto no Google Cloud Console.
# - Ative a API do Earth Engine nesse projeto.
# - Substitua 'thermal-luizadraeger' pelo ID do seu projeto.
# - Durante a execução, será solicitado um token de autenticação no Colab.

from google.colab import drive

# Monta o Google Drive no ambiente do Colab
# force_remount=True garante remontagem caso já esteja conectado
try:
    drive.mount('/content/drive', force_remount=True)
except:
    drive.mount('/content/drive')

# ─── Autenticação no Google Earth Engine ─────────────────────────────────────
# Abre um fluxo de login para autorizar o acesso ao EE
import ee
ee.Authenticate()

# Inicializa o Earth Engine com o projeto especificado
# Substitua pelo seu próprio ID de projeto no Google Cloud
ee.Initialize(project='thermal-luizadraeger')

Mounted at /content/drive


In [4]:
def buildfooprint(lat, lon, radius_m):
    """
    Extrai footprints e alturas de edifícios usando Google Open Buildings.
    """
    try:
        point = ee.Geometry.Point([lon, lat])
        region = point.buffer(radius_m).bounds()

        # Dataset de Alturas (Temporal V1)
        temporal_col = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1') \
                         .filterBounds(region) \
                         .sort('system:time_start', False)
        latest_raster = temporal_col.first()

        # Polígonos de Footprints (V3)
        buildings_fc = ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons') \
                         .filterBounds(region)

        # Amostragem de alturas por polígono
        sampled_fc = latest_raster.select('building_height').reduceRegions(
            collection=buildings_fc,
            reducer=ee.Reducer.median(),
            scale=1.0
        )

        features = sampled_fc.getInfo().get('features', [])
        building_list = []

        for f in features:
            props = f.get('properties', {})
            geom = f.get('geometry')
            if not geom: continue

            h_val = props.get('median')
            # Define altura padrão de 3.5m caso não encontre o dado
            height = float(h_val) if (h_val and h_val > 0) else 3.5

            # Extração de coordenadas para o Grasshopper
            coords = geom['coordinates'][0] if geom['type'] == 'Polygon' else geom['coordinates'][0][0]

            building_list.append({
                "geometry": coords,
                "height": height
            })

        return building_list
    except Exception as e:
        raise Exception(f"Erro no processamento GEE: {str(e)}")

In [28]:
%cd /content
!rm -rf URBAN-DATA-ANALYTICS

/content


In [29]:
!git clone https://github.com/Luizadraeger/URBAN-DATA-ANALYTICS.git

Cloning into 'URBAN-DATA-ANALYTICS'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 70 (delta 23), reused 6 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 154.53 KiB | 4.42 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [32]:
from shapely.geometry import Point
def get_river_data(lat, lon, radius_m):
    """
    Processa o recorte do Rio Paquequer e retorna os dados para o Grasshopper.
    """
    base_path = '/content/URBAN-DATA-ANALYTICS/references'

    # 1. Localização dinâmica da pasta (imune a erros de caracteres especiais)
    if not os.path.exists(base_path):
        return {"error": "Pasta 'references' não encontrada. Verifique o git clone."}

    target_folder = None
    for f in os.listdir(base_path):
        if "Hidrografia" in f or "Paquequer" in f:
            target_folder = os.path.join(base_path, f)
            break

    if not target_folder:
        return {"error": "Pasta de hidrografia não encontrada."}

    try:
        # 2. Carregamento do Shapefile
        shp_file = next((f for f in os.listdir(target_folder) if f.endswith('.shp')), None)
        if not shp_file:
            return {"error": "Arquivo .shp não encontrado."}

        full_path = os.path.join(target_folder, shp_file)
        gdf = gpd.read_file(full_path)

        if gdf.crs != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')

        # 3. Recorte Espacial (Clip)
        center_pt = Point(lon, lat)
        buffer_deg = radius_m / 111320.0 # Conversão aproximada para Teresópolis
        aoi_geom = center_pt.buffer(buffer_deg)

        river_clip = gdf[gdf.intersects(aoi_geom)].copy()

        # 4. Formatação para o Grasshopper (Transforma em lista de coordenadas)
        river_output = []

        if not river_clip.empty:
            for _, row in river_clip.iterrows():
                geom = row.geometry
                # Trata LineString e MultiLineString para que o Rhino entenda
                if geom.geom_type == 'LineString':
                    river_output.append({
                        "type": "LineString",
                        "coordinates": list(geom.coords)
                    })
                elif geom.geom_type == 'MultiLineString':
                    for line in geom.geoms:
                        river_output.append({
                            "type": "LineString",
                            "coordinates": list(line.coords)
                        })

            print(f"✅ {len(river_output)} segmentos de rio processados para Lat: {lat}")
            return river_output
        else:
            return [] # Retorna lista vazia se não houver rio no raio

    except Exception as e:
        print(f"Erro no processamento do Rio: {e}")
        return {"error": str(e)}

In [35]:
def get_terrain_data(lat, lon, radius_m):
    try:
        # Define o ponto e a área de busca
        point = ee.Geometry.Point([lon, lat])
        region = point.buffer(radius_m).bounds()

        # Carrega o dataset NASADEM (resolução de ~30m)
        dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation')

        # Amostragem de pontos dentro da ROI (escala de 30 metros para Teresópolis)
        sample = dem.sampleRegions(
            collection=ee.FeatureCollection([region]),
            scale=30,
            geometries=True
        )

        features = sample.getInfo().get('features', [])
        terrain_pts = []

        for f in features:
            terrain_pts.append({
                "type": "TerrainPoint",
                "coordinates": f['geometry']['coordinates'], # [lon, lat]
                "elevation": float(f['properties']['elevation'])
            })

        print(f"✓ Relevo: {len(terrain_pts)} pontos de altitude coletados.")
        return terrain_pts
    except Exception as e:
        print(f"⚠️ Erro no Pipeline de Relevo: {e}")
        return []